[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skarma91/logicmojo-ai-july-2026/blob/main/modules/module-2-ml-dl-essentials/05-training-in-practice/micro-assignment/assignment.ipynb)

# Micro-assignment 2.5: Training in practice

Fill in each cell. Match the expected output in `README.md`. PyTorch only.

### Problem 1: A DataLoader batch

In [18]:
import torch
from torch.utils.data import TensorDataset, DataLoader
X = torch.randn(100, 8)
y = torch.randint(0, 3, (100,))
# Wrap in a TensorDataset + DataLoader (batch_size=16), take the first batch, print X and y shapes.
# your code here
dataset = TensorDataset(X, y)

data = DataLoader(
    dataset,
    batch_size=16
)

X_batch, y_batch = next(iter(data))

print("X shape:", X_batch.shape)
print("y shape:", y_batch.shape)

X shape: torch.Size([16, 8])
y shape: torch.Size([16])


### Problem 2: Train and evaluate

In [54]:
# Load load_digits, split off 25% test (random_state=0, stratify=y), scale on train,
# set torch.manual_seed(0), train Linear(64,32)-ReLU-Linear(32,10) with CrossEntropyLoss and
# Adam(lr=0.01) for 150 steps, then print whether the test accuracy exceeds 0.90.
# your code here
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import float32, tensor
import torch.nn as nn 

dataset = load_digits()
x = dataset.data
y = dataset.target

X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.25, random_state=0, stratify=y
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = tensor(X_train, dtype=float32)
X_test = tensor(X_test, dtype=float32)

y_train = tensor(y_train, dtype=torch.long)
y_test = tensor(y_test, dtype=torch.long)

torch.manual_seed(0)

model = nn.Sequential(
    nn.Linear(64,32),
    nn.ReLU(),
    nn.Linear(32,10)
)

loss_fn = nn.CrossEntropyLoss()

adam = torch.optim.Adam(model.parameters(), lr=0.01, )

for _ in range(150):
    prediction = model(X_train)

    loss = loss_fn(prediction, y_train)

    adam.zero_grad()

    loss.backward()

    adam.step()

model.eval()

with torch.no_grad():
    prediction = model(X_test)
    pred_class = prediction.argmax(dim=1)
    accuracy = (pred_class == y_test).float().mean()

print(f"Test accuracy: {accuracy.item():.4f}")
print(f"Accuracy exceeds 0.90: {accuracy.item() > 0.90}")

Test accuracy: 0.9711

Accuracy exceeds 0.90: True

### Problem 3: Save and reload give identical predictions

In [67]:
# Create a model, save its state_dict, load into a fresh model of the same shape,
# and print whether the two give identical predictions on a batch.
# your code here
import torch.nn as nn

model_1 = nn.Sequential(
    nn.Linear(8, 32),
    nn.ReLU(),
    nn.Linear(32,3)
)

X = torch.randn(16, 8)

model_1.eval()

with torch.no_grad():
    pred1 = model_1(X)

torch.save(model_1.state_dict(), "model_1_saved.pth")

model_2 = nn.Sequential(
    nn.Linear(8, 32),
    nn.ReLU(),
    nn.Linear(32,3)
)

model_2.load_state_dict(
    torch.load('model_1_saved.pth', weights_only=True)
)

model_2.eval()
with torch.no_grad():
    pred2 = model_2(X)

print(torch.equal(pred1, pred2))

True

### Problem 4: Read the overfitting onset

In [72]:
val_losses = [0.90, 0.61, 0.48, 0.52, 0.70]
# Print the epoch index with the lowest validation loss.
# your code here
best_epoch = torch.argmin(torch.tensor(val_losses))
print(f"best epoch: {best_epoch}")

best epoch: 2

### Problem 5: A mismatch is a RuntimeError

In [95]:
# Trigger a shape mismatch with an incompatible matmul, catch it and print the error type,
# then fix the shapes and print the working result shape. Add the one-line note about device mismatch.
# your code here
mat1 = torch.randint(0, 9, (3,5))
mat2 = torch.randint(0, 9, (2, 2))

try:
    matmul = mat1 @ mat2
except RuntimeError as e:
    print(f"error type:, {type(e).__name__}")

mat1 = torch.randint(0, 9, (2, 2))
result = mat1 @ mat2 

print("After fix, shape:", result.shape)
print("note: a device mismatch (model on GPU, batch on CPU) raises the same RuntimeError; the fix is to move both to the same device")

error type:, RuntimeError

After fix, shape:
torch.Size([2, 2])

note: a device mismatch (model on GPU, batch on CPU) raises the same RuntimeError; the fix is to move both to the 
same device

### Problem 6: Reproducibility

In [103]:
# Write a helper that sets torch.manual_seed(seed) and returns float(torch.rand(1)).
# Call it twice with the same seed and print whether the values match.
# your code here
def random_no(seed):
    torch.manual_seed(seed)
    return float(torch.rand(1))

try1 = random_no(42)
try2 = random_no(42)

print(f"Runs match: {try1 == try2}")

Runs match: True